In [5]:
!python3 cli/finance.py -ti AAPL -fi data/aapl.csv

#### Fama-French dataset

[[1]](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html) Dartmouth College, French, K.R (2025) *Current Research Returns* Available at: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html (Accessed: 17 December 2025)

In [59]:
from pandas import DataFrame, read_csv

path: str = "data/F-F_Research_Data_5_Factors_2x3_daily.csv"
fama_french: DataFrame = read_csv(path)

fama_french.index = fama_french.index.astype("int32")

for column in fama_french.columns:
  min: float = fama_french[column].min()
  max: float = fama_french[column].max()
  ran: float = max - min

  print(f"{column:->8} |min: {min:>6} |max: {max:>6} |range: {ran:>6.3f}")

--Mkt-RF |min: -17.44 |max:  11.36 |range: 28.800
-----SMB |min: -11.15 |max:   6.08 |range: 17.230
-----HML |min:  -5.03 |max:   6.73 |range: 11.760
-----RMW |min:  -2.97 |max:   4.57 |range:  7.540
-----CMA |min:  -5.31 |max:   2.48 |range:  7.790
------RF |min:    0.0 |max:   0.06 |range:  0.060


In [ ]:
from pandas import DataFrame, Series, read_csv, to_datetime

path: str = "data/aapl.csv"
ticker: DataFrame = read_csv(path)
ticker.index = to_datetime(ticker.index, utc = True).strftime("%Y%m%d")
ticker.index = ticker.index.astype("int32")

returns: Series = ticker["Close"].pct_change() * 100
returns.dropna(inplace = True)

In [6]:
from pandas import merge, DataFrame

data: DataFrame = merge(returns, fama_french, how = "inner", left_index = True, right_index = True)
data.to_csv("data/fama_french_data.csv", index = False)

In [22]:
# Metrics declaration
import numpy as np

class Metrics():
  @classmethod
  def mean_squared(cls, error: np.ndarray) -> float:
    m: int = error.shape[0]
    return ((error.T @ error) * (1 / m)).item()

  @classmethod
  def root_mean_squared(cls, error: np.ndarray) -> float:
    return np.sqrt(cls.mean_squared(error))

  @classmethod
  def r_squared(cls, error: np.ndarray, variance: np.ndarray) -> float:
    return (1 - ((error.T @ error) / (variance.T @ variance))).item()

In [20]:
import numpy as np

class Multi_Linear_Regression:
  C: np.ndarray 
  M: np.ndarray 

  def train(self, x_train: np.ndarray, y_train: np.ndarray, lr: float = 1e-5, epoch: int = 1000) -> None:
    m, n = x_train.shape[0], x_train.shape[1]

    self.M: np.ndarray = np.ones((n, 1))
    self.C: np.ndarray = np.ones((1, 1))
    for _ in range(epoch): 
      y_hat: np.ndarray = (x_train @ self.M) + self.C
      error: np.ndarray = y_hat - y_train

      M_der: np.ndarray = (x_train.T @ error) * (2 / m)
      C_der: np.ndarray = (np.ones(m) @ error) * (2 / m)

      self.C: np.ndarray = self.C - (lr * C_der)
      self.M: np.ndarray = self.M - (lr * M_der)
  
  def predict(self, x_test: np.ndarray) -> np.ndarray:
    return (x_test @ self.M) + self.C

In [47]:
from pandas import DataFrame, read_csv
from sklearn.model_selection import TimeSeriesSplit
import numpy as np

data: DataFrame = read_csv("data/fama_french_data.csv")
x: DataFrame = data.iloc[:, 1:]
y: DataFrame = data[["Close"]]

split: TimeSeriesSplit = TimeSeriesSplit(n_splits = 5)

for fold, (train_index, test_index) in enumerate(split.split(x)):
  x_train: np.ndarray = x.iloc[train_index].values
  x_test: np.ndarray = x.iloc[test_index].values
  y_train: np.ndarray = y.iloc[train_index].values
  y_test: np.ndarray = y.iloc[test_index].values

  model: Multi_Linear_Regression = Multi_Linear_Regression()
  model.train(x_train, y_train, lr = 2e-3,epoch = 100000)  

  y_hat: np.ndarray = model.predict(x_test)

  mean: np.ndarray = np.mean(y_test, dtype = np.ndarray)
  variance: np.ndarray = y_test - mean
  error: np.ndarray = y_hat - y_test

  print(f"Mean squared error: {Metrics.mean_squared(error)}")
  print(f"Root mean squared error: {Metrics.root_mean_squared(error)}")
  print(f"R2 score: {Metrics.r_squared(error, variance)}")

Mean squared error: 5.739709759777465
Root mean squared error: 2.3957691374123393
R2 score: 0.15749582335627021
Mean squared error: 12.670131865605837
Root mean squared error: 3.559512869144574
R2 score: 0.169542273528704
Mean squared error: 4.522659297225277
Root mean squared error: 2.126654484683696
R2 score: 0.29843124776883
Mean squared error: 1.712280762490842
Root mean squared error: 1.3085414638026729
R2 score: 0.310825194192894
Mean squared error: 2.0775283304716368
Root mean squared error: 1.4413633582381775
R2 score: 0.462133490457995


In [46]:
import numpy as np
from pandas import read_csv, DataFrame
import json

data: DataFrame = read_csv("data/fama_french_data.csv")
x: np.ndarray = data.iloc[:, 1:].values
y: np.ndarray = data[["Close"]].values

model: Multi_Linear_Regression = Multi_Linear_Regression()
model.train(x, y) 

model_data: dict = {
  "C": float(model.C),
  "M": model.M.tolist()
}

with open("fama_french_parameters.json", "w") as file:
  json.dump(model_data, file)

/tmp/ipykernel_13187/1366057297.py:13: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "C": float(model.C),
